# Modern Data Stack: dbt — prototyping in DuckDB

The pattern: prototype your SQL against DuckDB here, then paste each SELECT into a
dbt model file. The final section scaffolds the dbt project files for you.
Requires `pip install duckdb dbt-duckdb`.

## 1. Load raw CSV into DuckDB

In [ ]:
import duckdb
import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 1000
raw_orders = pd.DataFrame({
    "ORDER_ID": range(1, n + 1),
    "CustomerId": rng.integers(1, 200, n),
    "order_TS": pd.Timestamp("2026-06-01") + pd.to_timedelta(rng.integers(0, 30 * 24, n), unit="h"),
    "AMT": rng.uniform(5, 300, n).round(2),
    "STATUS": rng.choice(["completed", "returned", "pending"], n, p=[0.85, 0.05, 0.10]),
})
raw_orders.to_csv("raw_orders.csv", index=False)   # messy names on purpose

con = duckdb.connect("dbt_prototype.duckdb")
con.execute("CREATE OR REPLACE TABLE raw_orders AS SELECT * FROM read_csv_auto('raw_orders.csv')")
con.sql("SELECT * FROM raw_orders LIMIT 3").df()

## 2. Prototype the STAGING model: rename, cast, clean

In [ ]:
STG_ORDERS = """
SELECT
    ORDER_ID                 AS order_id,
    CustomerId               AS customer_id,
    CAST(order_TS AS DATE)   AS order_date,
    AMT                      AS amount,
    LOWER(STATUS)            AS status
FROM raw_orders
"""
con.execute(f"CREATE OR REPLACE VIEW stg_orders AS {STG_ORDERS}")
con.sql("SELECT * FROM stg_orders LIMIT 3").df()

## 3. Prototype INTERMEDIATE and MART models on top (this is what ref() chains)

In [ ]:
INT_COMPLETED = """
SELECT * FROM stg_orders WHERE status = 'completed'
"""
con.execute(f"CREATE OR REPLACE VIEW int_orders_completed AS {INT_COMPLETED}")

MART_REVENUE = """
SELECT
    order_date,
    COUNT(*)                    AS n_orders,
    ROUND(SUM(amount), 2)       AS revenue,
    ROUND(AVG(amount), 2)       AS avg_order_value
FROM int_orders_completed
GROUP BY order_date
ORDER BY order_date
"""
con.execute(f"CREATE OR REPLACE VIEW mart_revenue_daily AS {MART_REVENUE}")
con.sql("SELECT * FROM mart_revenue_daily LIMIT 5").df()

## 4. Sanity checks — these become dbt tests

In [ ]:
checks = {
    "unique order_id":       "SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM stg_orders",
    "not_null customer_id":  "SELECT COUNT(*) FROM stg_orders WHERE customer_id IS NULL",
    "accepted status values":"SELECT COUNT(*) FROM stg_orders WHERE status NOT IN ('completed','returned','pending')",
    "no negative amounts":   "SELECT COUNT(*) FROM stg_orders WHERE amount < 0",
}
for name, sql in checks.items():
    violations = con.execute(sql).fetchone()[0]
    print(f"{'PASS' if violations == 0 else 'FAIL'}  {name} ({violations} violations)")

## 5. Scaffold the actual dbt project from these SELECTs

In [ ]:
import os
import pathlib

proj = pathlib.Path("../dbt_project")
(proj / "models" / "staging").mkdir(parents=True, exist_ok=True)
(proj / "models" / "marts").mkdir(parents=True, exist_ok=True)

(proj / "dbt_project.yml").write_text(
    "name: learning_dbt\nversion: '1.0'\nprofile: learning_dbt\n"
    "models:\n  learning_dbt:\n    staging:\n      +materialized: view\n"
    "    marts:\n      +materialized: table\n")

(proj / "profiles.yml").write_text(
    "learning_dbt:\n  target: dev\n  outputs:\n    dev:\n"
    "      type: duckdb\n      path: dev.duckdb\n")

(proj / "models" / "staging" / "stg_orders.sql").write_text(
    STG_ORDERS.replace("raw_orders", "{{ source('shop', 'raw_orders') }}"))
(proj / "models" / "staging" / "sources.yml").write_text(
    "version: 2\nsources:\n  - name: shop\n    tables:\n      - name: raw_orders\n")
(proj / "models" / "staging" / "schema.yml").write_text(
    "version: 2\nmodels:\n  - name: stg_orders\n    columns:\n"
    "      - name: order_id\n        data_tests: [unique, not_null]\n"
    "      - name: status\n        data_tests:\n"
    "          - accepted_values:\n              values: [completed, returned, pending]\n")
(proj / "models" / "marts" / "mart_revenue_daily.sql").write_text(
    MART_REVENUE.replace("int_orders_completed",
                         "{{ ref('stg_orders') }} WHERE status = 'completed'"))

print("dbt project scaffolded at ../dbt_project — run:")
print("  cd ../dbt_project && dbt run && dbt test && dbt docs generate")

## 6. Mini-project notes

After `dbt run && dbt test` pass, run `dbt docs generate && dbt docs serve` and
describe the lineage graph here (source → staging → mart). Then load the raw CSV
into the dbt DuckDB (`dev.duckdb`) or point the source at this notebook's database.